# 02 — Feature Engineering

This notebook walks through the feature engineering step of the credit risk scorecard pipeline.  
We create derived predictors, handle data type conversions, and cap outliers.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src.feature_engineering import (
    cap_outliers, convert_int_rate, convert_emp_length,
    convert_term, engineer_features
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
%matplotlib inline

## 2.1 Load Cleaned Data

In [ ]:
df = pd.read_csv(config.CLEANED_CSV)
print(f"Shape: {df.shape}")
df.head()

## 2.2 Interest Rate Conversion

The `int_rate` column may contain `%` symbols. We strip them and convert to float.

In [ ]:
print(f"Before: dtype={df['int_rate'].dtype}, sample={df['int_rate'].iloc[:3].tolist()}")
df["int_rate"] = convert_int_rate(df["int_rate"])
print(f"After:  dtype={df['int_rate'].dtype}, sample={df['int_rate'].iloc[:3].tolist()}")

df["int_rate"].hist(bins=40, color="#3498db", edgecolor="white", figsize=(10, 4))
plt.title("Interest Rate Distribution (cleaned)", fontweight="bold")
plt.xlabel("Interest Rate (%)")
plt.show()

## 2.3 DTI – Capping Outliers at 99th Percentile

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df["dti"].dropna(), bins=50, color="#e74c3c", edgecolor="white")
axes[0].set_title("DTI – Before Capping")

df["dti"] = cap_outliers(df["dti"], upper_pct=99)

axes[1].hist(df["dti"].dropna(), bins=50, color="#2ecc71", edgecolor="white")
axes[1].set_title("DTI – After Capping (99th)")

plt.tight_layout()
plt.show()
print(f"DTI range: {df['dti'].min():.2f} – {df['dti'].max():.2f}")

## 2.4 Engineered Features

In [ ]:
# Reload and run full engineering
df = pd.read_csv(config.CLEANED_CSV)
df = engineer_features(df)
print(f"Engineered shape: {df.shape}")
df.head()

## 2.5 New Feature Distributions

In [ ]:
new_features = ["payment_to_income", "fico_midpoint", "loan_to_income",
                "emp_length_numeric", "annual_inc_log", "delinq_flag"]
new_features = [c for c in new_features if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.ravel(), new_features):
    df[col].hist(ax=ax, bins=40, color="#9b59b6", edgecolor="white", alpha=0.85)
    ax.set_title(col, fontsize=12, fontweight="bold")

plt.suptitle("Engineered Feature Distributions", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 2.6 Correlation with Target

In [ ]:
target_corr = df.select_dtypes(include="number").corr()["bad_loan"].drop("bad_loan").sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in target_corr]
target_corr.plot.barh(ax=ax, color=colors)
ax.set_title("Correlation with bad_loan", fontsize=14, fontweight="bold")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

## 2.7 Summary

| Feature | Description |
|---------|-------------|
| `payment_to_income` | Monthly installment / monthly income |
| `delinq_flag` | 1 if any delinquency in last 2 years |
| `fico_midpoint` | Average of FICO range boundaries |
| `loan_to_income` | Loan amount / annual income |
| `emp_length_numeric` | Employment years (0–10 scale) |
| `annual_inc_log` | Log-transformed annual income |

Proceed to **03_woe_iv_binning.ipynb** →